In [ ]:
# --- config ---
IMAGE_DIR = "../data/photos"      # or a folder of your own photos
MODEL_NAME = "clip-ViT-B-32"
MAX_SIDE = 512                   # shrink on load; CLIP only sees 224 x 224 anyway

In [ ]:
# --- load + embed ---
from pathlib import Path
from PIL import Image, ImageOps
from sentence_transformers import SentenceTransformer
import numpy as np

def load(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    return img

paths = sorted(p for p in Path(IMAGE_DIR).iterdir()
               if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
images = [load(p) for p in paths]

model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(images, batch_size=32, normalize_embeddings=True,
                          show_progress_bar=True)
print(f"{len(paths)} images, {embeddings.shape[1]} dimensions")

In [ ]:
# --- search by text ---
import matplotlib.pyplot as plt

QUERY = "the ocean"
TOP_K = 3

scores = np.dot(embeddings, model.encode(QUERY, normalize_embeddings=True))
best = np.argsort(-scores)[:TOP_K]

fig, axes = plt.subplots(1, TOP_K, figsize=(4 * TOP_K, 4))
for ax, i in zip(np.atleast_1d(axes), best):
    ax.imshow(images[i])
    ax.set_title(f"{scores[i]:.3f}  {paths[i].name}", fontsize=9)
    ax.axis("off")
fig.suptitle(QUERY)
plt.show()

In [ ]:
# --- which description fits which image? (for a handful of images) ---
import pandas as pd

PROMPTS = [   # edit to match your six images
    "a photo of a castle",
    "a photo of a beach",
    "a photo of mountains",
    "a photo of sheep",
    "a photo of a bridge",
    "My lovely pet",
]

prompt_embeddings = model.encode(PROMPTS, normalize_embeddings=True)
matrix = np.dot(prompt_embeddings, embeddings.T)

fig, axes = plt.subplots(1, len(images), figsize=(2.5 * len(images), 2.5))
for ax, img, p in zip(np.atleast_1d(axes), images, paths):
    ax.imshow(img)
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")
plt.show()

display(pd.DataFrame(matrix, index=PROMPTS, columns=[p.name for p in paths])
        .style.background_gradient(axis=None).format("{:.3f}"))